# Ridge and kernel-ridge experiments
Original coursework experiments. The supplied train.csv/test.csv are not bundled. Use examples/compare_regressors.py for the fully reproducible synthetic experiment with preprocessing fitted inside each CV fold.


### Experiment cell 1


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import mean_squared_error

# λ grid
lambdas = np.arange(0.02, 0.32, 0.02)

# store results
ridge_star_errors = []   # errors for β_{λ*}
ols_errors = []          # OLS errors


for seed in range(10):
    np.random.seed(seed)

    n_samples, n_features = 200, 50

    # training data
    X = np.random.randn(n_samples, n_features)
    true_coef = np.random.randn(n_features)
    Y = X @ true_coef + np.random.randn(n_samples) * 0.5

    # test data
    X_test = np.random.randn(50, n_features)
    Y_test = X_test @ true_coef + np.random.randn(50) * 0.5

    # split into S1 and S2
    X1, X2 = X[:100], X[100:]
    Y1, Y2 = Y[:100], Y[100:]

    # validation errors for each λ
    val_errors = []

    for lam in lambdas:
        model = Ridge(alpha=100 * lam, fit_intercept=False)
        model.fit(X1, Y1)

        pred = model.predict(X2)
        val_errors.append(mean_squared_error(Y2, pred))

    # select λ*
    lam_star = lambdas[np.argmin(val_errors)]

    # retrain on full data with λ*
    model = Ridge(alpha=200 * lam_star, fit_intercept=False)
    model.fit(X, Y)

    ridge_pred = model.predict(X_test)
    ridge_star_errors.append(mean_squared_error(Y_test, ridge_pred))

    # OLS
    ols = LinearRegression(fit_intercept=False)
    ols.fit(X, Y)

    ols_pred = ols.predict(X_test)
    ols_errors.append(mean_squared_error(Y_test, ols_pred))



avg_ridge_star = np.mean(ridge_star_errors)
avg_ols = np.mean(ols_errors)

print("Average test error (Ridge with CV λ*):", avg_ridge_star)
print("Average test error (OLS):", avg_ols)



lambda_test_errors = []

for lam in lambdas:
    errs = []

    for seed in range(10):
        np.random.seed(seed)

        n_samples, n_features = 200, 50

        X = np.random.randn(n_samples, n_features)
        true_coef = np.random.randn(n_features)
        Y = X @ true_coef + np.random.randn(n_samples) * 0.5

        X_test = np.random.randn(50, n_features)
        Y_test = X_test @ true_coef + np.random.randn(50) * 0.5

        model = Ridge(alpha=200 * lam, fit_intercept=False)
        model.fit(X, Y)

        pred = model.predict(X_test)
        errs.append(mean_squared_error(Y_test, pred))

    lambda_test_errors.append(np.mean(errs))

lambda_test_errors = np.array(lambda_test_errors)


plt.figure()

# curve: Ridge vs λ
plt.plot(lambdas, lambda_test_errors, marker='o', label='Ridge (varying λ)')

# constant lines
plt.axhline(avg_ridge_star, linestyle='--', label='Ridge (CV λ*)')
plt.axhline(avg_ols, linestyle='--', label='OLS')

plt.xlabel("λ")
plt.ylabel("Average Test MSE")
plt.title("Ridge Regression with Cross-Validation")
plt.legend()

plt.show()

### Experiment cell 2


In [ ]:
import numpy as np
test_data = np.loadtxt("test.csv", delimiter = "," )
train_data = np.loadtxt("train.csv", delimiter = "," )


X_train = train_data[:, :21]
y_train = train_data[:, 21]

X_test = test_data[:, :21]
y_test = test_data[:, 21]

### Experiment cell 3


In [ ]:
from sklearn.linear_model import Ridge
from sklearn.kernel_ridge import KernelRidge

# Ridge Regression
modelR = Ridge()

# Polynomial Ridge Regression
modelPR = KernelRidge(kernel="polynomial", degree=2)

# Gaussian Kernel Ridge Regression
modelG = KernelRidge(kernel="rbf")

### Experiment cell 4


In [ ]:

# Fit the model
modelR.fit(X_train, y_train)


y_train_pred = modelR.predict(X_train)
y_test_pred = modelR.predict(X_test)

print(mean_squared_error(y_train, y_train_pred))
print(mean_squared_error(y_test, y_test_pred))

### Experiment cell 5


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.kernel_ridge import KernelRidge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error
from sklearn import preprocessing

# Load data
train_data = np.loadtxt("train.csv", delimiter=",")
test_data = np.loadtxt("test.csv", delimiter=",")

X_train_full = train_data[:, :21]
Y_train_full = train_data[:, 21]
X_test = test_data[:, :21]
Y_test = test_data[:, 21]

training_sizes = range(4000, 11000, 1000)
n_repeats = 5

mse_results = {
    "Ridge": [],
    "Polynomial Ridge": [],
    "Kernel Ridge": []
}

for size in training_sizes:
    mse_per_method = {name: [] for name in mse_results}

    for _ in range(n_repeats):
        # Randomly sample 'size' examples
        indices = np.random.choice(len(X_train_full), size=size, replace=False)
        X_sample = X_train_full[indices]
        Y_sample = Y_train_full[indices]

        # Scale data (fit on training sample, apply to both)
        scaler = preprocessing.StandardScaler().fit(X_sample)
        X_scaled = scaler.transform(X_sample)
        X_test_scaled = scaler.transform(X_test)

        # Regularization values scaled by training size n
        n = size
        alphas = np.array([1e-5, 1e-4, 1e-3, 1e-2, 0.1, 1.0, 10.0]) * n
        param_grid = {'alpha': alphas}

        models = {
            "Ridge": Ridge(),
            "Polynomial Ridge": KernelRidge(kernel="polynomial", degree=2),
            "Kernel Ridge": KernelRidge(kernel="rbf")
        }

        for name, model in models.items():
            # 2-fold cross-validation to find best alpha
            grid_search = GridSearchCV(model, param_grid, cv=2, scoring="neg_mean_squared_error")
            grid_search.fit(X_scaled, Y_sample)
            best_estimator = grid_search.best_estimator_
            y_pred = best_estimator.predict(X_test_scaled)
            mse = mean_squared_error(Y_test, y_pred)
            mse_per_method[name].append(mse)

    for name in mse_results:
        mse_results[name].append(np.mean(mse_per_method[name]))

# Plot
plt.figure(figsize=(10, 6))
for name, mse_values in mse_results.items():
    plt.plot(list(training_sizes), mse_values, marker='o', label=name)

plt.xlabel("Training Data Size")
plt.ylabel("Average MSE (Test Set)")
plt.title("Average MSE vs Training Data Size (with CV Regularization)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("mse_vs_training_size.png", dpi=150)
plt.show()